# Delta Lake Optimization Project

This notebook is a guided, hands‑on lab to explore core Delta Lake optimization techniques in Databricks. You'll iteratively: generate data → profile baseline performance → apply optimizations → observe impact in the Spark UI and table metadata.

## Learning Objectives
By the end you should be able to:
- Select appropriate physical layout strategies (partitioning, Z-Ordering, Liquid Clustering)
- Diagnose small-file and data-skipping issues
- Apply manual vs automatic compaction
- Use VACUUM safely for storage hygiene
- Read/interpret Spark UI metrics (files scanned, data read, predicate pushdown, scan time)

## Flow Overview
1. Environment + table name registry
2. Data generation (idempotent)
3. Comparing Partitioning, Z-Ordering, Liquid Clustering
  - Baseline query
  - Partitioning impact
  - Z-Ordering for multi-dimensional skipping
  - Liquid Clustering (adaptive layout)
4. Handling small and orphaned files
  - Manual compaction
  - Auto compaction 
  - VACUUM lifecycle management
5. Cleanup

> Tip: After each transformation, open the Spark UI and record: files scanned, total data read MB, and wall-clock time. Building a mini table of results reinforces the concepts.

## Setup
We first define the catalog & schema (database) and then a centralized registry of logical table names to keep code DRY and readable. Ensure your workspace permissions allow catalog/schema creation.

In [0]:
# Configuration
CATALOG_NAME = "delta_optimization_project"
SCHEMA_NAME = "sales_data"

# Create the catalog and schema if they don't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

DataFrame[]

In [0]:
# Centralized table name registry & helper utilities
TABLES = {
    "raw": f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_raw",
    "partitioned": f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_partitioned",
    "raw_zorder": f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_raw_zorder",
    "to_compact": f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_to_compact",
    "auto_compact": f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_auto_compact",
    "liquid_clustered": f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_liquid_clustered",
}

def tbl(key: str) -> str:
    """Return fully qualified table name by logical key."""
    return TABLES[key]

from typing import Optional
import pyspark.sql.functions as F
from pyspark.sql.functions import col

## Preparation: Generate Synthetic Sales Data

We create a moderately large, semi-realistic sales fact dataset to exercise partition pruning and data skipping. Characteristics:
- 50M rows (adjustable) to give enough files for optimization to matter
- Mixed cardinalities: `country` (low), `customer_id` (medium/high), `product_id` (medium) for contrasting layout strategies
- Temporal column `sale_date` enabling potential future partitioning or clustering experiments

### Design Notes
- Data generation is idempotent: if the base table exists we skip regeneration (saves cluster time)
- We intentionally over-partition the initial write (`num_partitions` + low `maxRecordsPerFile`) to create many small files and highlight gains from compaction

### Your Tasks
1. Run the cell.
2. If generation runs, note: number of partitions written, approximate file count (later visible via `DESCRIBE DETAIL` or storage listing).
3. If skipped, confirm reuse message printed.

Proceed once the table exists.

In [0]:
# Generate synthetic data at scale using Spark (executor-friendly, no pandas/Faker)
import pyspark.sql.functions as F

# If the target table already exists, skip expensive generation and reuse it.
# This makes the notebook idempotent for iterative runs.
raw_exists = spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{SCHEMA_NAME} LIKE 'sales_raw'").count() > 0
if raw_exists:
    print(f"Table {tbl('raw')} already exists; skipping data generation.")
else:
    # Number of records to generate (adjust for your cluster)
    num_records = 50_000_000

    # Controls the number of output files (one file per output partition, roughly).
    # Increase partitions to create more small files; decrease to reduce file count.
    num_partitions = 1000

    # Small list of countries used for sampling (kept on driver only)
    countries = ['United States','United Kingdom','Germany','France','Canada','Australia','India','Brazil','Japan','Netherlands']

    # Build the DataFrame using Spark primitives (scales across executors)
    df = (
        spark.range(num_records)
        .repartition(num_partitions)
        .withColumnRenamed('id', 'seq')
        .withColumn(
            'transaction_id',
            F.concat(
                F.lit('txn_'),
                F.col('seq').cast('string'),
                F.lit('_'),
                # cast the double returned by rand() to string before md5 to avoid datatype mismatch
                F.substring(F.md5(F.rand(12345).cast('string')), 1, 8)
            )
        )
        .withColumn('customer_id', (F.floor(F.rand(42) * 1001) + 1000).cast('int'))
        .withColumn('product_id', (F.floor(F.rand(99) * 401) + 100).cast('int'))
        .withColumn('sale_date', F.date_sub(F.current_date(), F.floor(F.rand(7) * 730).cast('int')))
        .withColumn('quantity', (F.floor(F.rand(11) * 10) + 1).cast('int'))
        .withColumn('unit_price', F.round(F.rand(13) * 190.0 + F.lit(10.5), 2))
        .withColumn(
            'country',
            F.element_at(
                F.array(*[F.lit(c) for c in countries]),
                (F.floor(F.rand(21) * len(countries)) + 1).cast('int')
            )
        )
    )
    # Write the data to a Delta table. Use maxRecordsPerFile to force small files in a performant, distributed write.
    (df.write
       .format('delta')
       .option('maxRecordsPerFile', 5000)
       .mode('overwrite')
       .saveAsTable(tbl('raw'))
    )

In [0]:
df_raw = spark.read.table(tbl('raw'))

# Part 1: Compare Partitioning, Z-Order, and Liquid Clustering

## Baseline Query (Unoptimized Table)

We establish a performance baseline before altering layout. This isolates the effect of later optimizations.

### Query Pattern
Filter on a specific `(customer_id, product_id)` pair. This mimics a targeted lookup often seen in downstream analytics or service patterns.

### What to Capture (Create a simple log table for yourself)
| Metric | Value | How to Find |
|--------|-------|-------------|
| Files scanned | ? | Spark UI -> SQL/Dataframe -> Scan node |
| Bytes read | ? | Spark UI -> Scan details |
| Duration (s) | ? | Job / Stage timeline |

Run the cell below to capture baseline

In [0]:
df_raw.where(
    (col("customer_id") == 1500)
    & (col("product_id") == 250)
    & (col("country") == "United States")
).show()

+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
|     seq|      transaction_id|customer_id|product_id| sale_date|quantity|unit_price|      country|
+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
| 6730139|txn_6730139_f9204462|       1500|       250|2024-07-23|       7|    164.58|United States|
|33713731|txn_33713731_2200...|       1500|       250|2025-05-13|       9|    136.78|United States|
|36390214|txn_36390214_4c9a...|       1500|       250|2025-10-21|       9|    120.45|United States|
|12396912|txn_12396912_3aae...|       1500|       250|2025-07-05|       5|     78.44|United States|
|37080913|txn_37080913_ad48...|       1500|       250|2024-11-11|       5|    121.25|United States|
| 7272378|txn_7272378_36f1a84f|       1500|       250|2025-04-26|       8|     48.89|United States|
|47799373|txn_47799373_a4a5...|       1500|       250|2025-10-10|       6|     38.77|United States|


1. Open Spark UI after it finishes.
2. Record metrics above.
3. Keep this snapshot; you'll compare after each optimization layer.

## Technique 1: Partitioning

Partitioning physically groups data into directory paths by one or more columns. Spark can then prune entire directories early when a query predicate fixes specific partition values, reducing file listing & scan work.

### What It Solves
- Efficient directory pruning for highly reused, low/moderate-cardinality filters
- Can bound per-query file counts for time/region sharded workloads

### When to Use
- Clear, consistent filtering patterns (e.g. date, region, category)
- Low cardinality columns (generally ≲ 1K distinct values across full dataset)
- Large tables where pruning meaningfully reduces scan

### When NOT to Use
- High-cardinality or rapidly growing distinct sets (user ids, fine-grain timestamps)
- Workloads without stable filter patterns
- Columns that mutate frequently (causes skew / small files)

### Trade-offs & Pitfalls
- Over-partitioning -> many small files + metadata overhead
- Too coarse partition -> limited pruning benefit
- Combining with Z-Order: partition first on coarse dimension (e.g. date) then Z-Order inside each partition for additional skipping on high-card columns

We choose `country` (~10 values) for demonstration.

Create a partitioned table

In [0]:
(df_raw.write.format("delta")
 .mode("overwrite")
 .partitionBy("country")
 .saveAsTable(tbl('partitioned')))

Run the same query on the partitioned table

In [0]:
(spark.read.table(tbl('partitioned'))
      .where(    
      (col("customer_id") == 1500) 
      & (col("product_id") == 250) 
      & (col("country") == 'United States')
      )
      .show())

+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
|     seq|      transaction_id|customer_id|product_id| sale_date|quantity|unit_price|      country|
+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
| 6730139|txn_6730139_f9204462|       1500|       250|2024-07-23|       7|    164.58|United States|
|33713731|txn_33713731_2200...|       1500|       250|2025-05-13|       9|    136.78|United States|
|36390214|txn_36390214_4c9a...|       1500|       250|2025-10-21|       9|    120.45|United States|
|12396912|txn_12396912_3aae...|       1500|       250|2025-07-05|       5|     78.44|United States|
|37080913|txn_37080913_ad48...|       1500|       250|2024-11-11|       5|    121.25|United States|
| 7272378|txn_7272378_36f1a84f|       1500|       250|2025-04-26|       8|     48.89|United States|
|47799373|txn_47799373_a4a5...|       1500|       250|2025-10-10|       6|     38.77|United States|


**Analysis of Partitioning:**

Go to the Spark UI and compare the execution plan with the baseline. You'll notice that a filter on the partition key (`country`) was present and Spark was able to prune entire directories, significantly reducing the amount of data scanned.

## Technique 2: Z-Order

Z-Ordering re-writes data files using a multi-dimensional space-filling curve (Z-order) so rows with similar values for selected columns co-locate. Think of it like organizing a library so related books (rows) live on the same shelf (file) — Spark can then skip whole shelves.

### What It Solves
- Improves data skipping without exploding partition counts
- Helps multi-column filtering patterns (e.g., customer_id AND product_id)
- Reduces number of files scanned for selective queries

### When to Use
- Multi-column filters in your queries
- Tables with 2–4 frequently queried, moderate/high-cardinality columns
- Data that doesn’t change constantly (batch / micro-batch growth)
- Need clustering on columns that would be poor physical partitions

### When NOT to Use
- Single-column selective queries (simple sorting / partitioning may suffice)
- Very frequent small incremental writes (rewrite overhead can outweigh benefit)
- More than 4 columns (diminishing returns & larger shuffle)
- Columns with extremely high entropy (e.g. UUID) where locality gain is minimal

### Column Choice Tips
| Scenario | Better Partition | Better Z-Order |
|----------|------------------|----------------|
| Very low cardinality | Yes | Usually unnecessary |
| High cardinality (but queried) | No | Often yes |
| Evolving / mixed predicates | Limited | Strong |
| Need adaptive maintenance | Sometimes | Consider Liquid Clustering |

We Z-Order on `country`, `customer_id`, `product_id` to boost skipping for the predicate used in our benchmark query.

> Operational Hint: Re-run Z-ORDER after significant (e.g. 10–20%) new data growth in the clustered dimensions or after large backfills.


Create Z-Ordered copy of raw table.

In [0]:
from delta.tables import *

spark.sql(f"DROP TABLE IF EXISTS {tbl('raw_zorder')}")

# Create a separate table from sales_raw to apply Z-Ordering to (keeps sales_partitioned untouched)
spark.sql(f"CREATE TABLE {tbl('raw_zorder')} USING DELTA AS SELECT * FROM {tbl('raw')}")

# Convert the new table to a DeltaTable object
delta_table = DeltaTable.forName(spark, tbl('raw_zorder'))

# Apply Z-Ordering on high-cardinality columns for the raw copy
delta_table.optimize().executeZOrderBy("country", "customer_id", "product_id")

DataFrame[path: string, metrics: struct<autoCompactParallelismStats:void,clusteringMetrics:void,clusteringStats:void,deletionVectorStats:struct<numDeletionVectorRowsRemoved:bigint,numDeletionVectorsRemoved:bigint>,endTimeMs:bigint,fileAgeHistogram:void,filesAdded:struct<avg:double,max:bigint,min:bigint,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<avg:double,max:bigint,min:bigint,totalFiles:bigint,totalSize:bigint>,numBatches:bigint,numBins:bigint,numBytesSkippedToReduceWriteAmplification:bigint,numFilesAdded:bigint,numFilesRemoved:bigint,numFilesSkippedToReduceWriteAmplification:bigint,numTableColumns:bigint,numTableColumnsWithStats:bigint,partitionsOptimized:bigint,preserveInsertionOrder:boolean,recompressionCodec:void,skippedArchivedFiles:bigint,startTimeMs:bigint,totalClusterParallelism:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,totalScheduledTasks:bigint,totalTaskExecutionTimeMs:bigint,zOrderStats:struct<inputCubeFiles:struct<num:bigint,size:bigint>,inpu

Rerun the query to see the effect of Z-Ordering on the z-ordered raw table

In [0]:
(spark.read.table(tbl('raw_zorder'))
      .where((col("customer_id") == 1500) & (col("product_id") == 250) & (col("country") == "United States"))
      .show())

+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
|     seq|      transaction_id|customer_id|product_id| sale_date|quantity|unit_price|      country|
+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
| 7272378|txn_7272378_36f1a84f|       1500|       250|2025-04-26|       8|     48.89|United States|
|12396912|txn_12396912_3aae...|       1500|       250|2025-07-05|       5|     78.44|United States|
|36390214|txn_36390214_4c9a...|       1500|       250|2025-10-21|       9|    120.45|United States|
|26505435|txn_26505435_b820...|       1500|       250|2025-11-25|       1|     94.01|United States|
|47476539|txn_47476539_c784...|       1500|       250|2025-08-07|       6|     43.38|United States|
| 6730139|txn_6730139_f9204462|       1500|       250|2024-07-23|       7|    164.58|United States|
|33713731|txn_33713731_2200...|       1500|       250|2025-05-13|       9|    136.78|United States|


**Analysis of Z-Ordering:**

In the Spark UI, look at the "Details" for the scan phase. You should see a significant reduction in the number of files read, demonstrating the effectiveness of data skipping.

## Technique 3: Liquid Clustering

Liquid Clustering dynamically reorganizes data based on query patterns, adapting to your workload without manual tuning. It decouples logical clustering from static partitions — Databricks maintains clustering as data evolves, avoiding rigid directory hierarchies while still enabling data skipping.

### Advantages vs Partitioning / Z-Order
- Adaptive: automatically reorganizes incremental data
- Multi-column "soft" clustering without explosion in partitions
- Plays well with schema evolution and changing query shapes

### When to Use
- Evolving query patterns
- Multiple clustering columns needed
- Want to replace both partitioning and Z-Ordering
- High churn / streaming upserts
- Many predicates across overlapping column sets
- Avoiding maintenance burden of periodic manual Z-Order runs

### When NOT to Use
- Simple, stable query patterns (a single partition key may be cheaper)
- Tables smaller than ~1GB where overhead outweighs benefit
- Workloads requiring deterministic physical ordering for downstream consumers

> Consider long-term ops: which technique minimizes recurring maintenance for your workload? Liquid Clustering can reduce hands-on maintenance for dynamic workloads but adds some runtime cost and operational monitoring requirements.


Create clustered table and load data.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {tbl('liquid_clustered')} (
  seq LONG,
  transaction_id STRING,
  customer_id INT,
  product_id INT,
  sale_date DATE,
  quantity INT,
  unit_price DOUBLE,
  country STRING
)
USING DELTA
CLUSTER BY (country, customer_id, product_id)
""")

# Insert data into the clustered table
df_raw.write.format("delta").mode("overwrite").saveAsTable(tbl('liquid_clustered'))

Run predicate query; record scan metrics. Compare with Z-Order table results.

In [0]:
# Rerun the query on the liquid clustered table
(spark.read.table(tbl('liquid_clustered'))
      .where((col("customer_id") == 1500) & (col("product_id") == 250) & (col("country") == "United States"))
      .show())

+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
|     seq|      transaction_id|customer_id|product_id| sale_date|quantity|unit_price|      country|
+--------+--------------------+-----------+----------+----------+--------+----------+-------------+
| 7272378|txn_7272378_36f1a84f|       1500|       250|2025-04-26|       8|     48.89|United States|
|12396912|txn_12396912_3aae...|       1500|       250|2025-07-05|       5|     78.44|United States|
|36390214|txn_36390214_4c9a...|       1500|       250|2025-10-21|       9|    120.45|United States|
|26505435|txn_26505435_b820...|       1500|       250|2025-11-25|       1|     94.01|United States|
|47476539|txn_47476539_c784...|       1500|       250|2025-08-07|       6|     43.38|United States|
| 6730139|txn_6730139_f9204462|       1500|       250|2024-07-23|       7|    164.58|United States|
|33713731|txn_33713731_2200...|       1500|       250|2025-05-13|       9|    136.78|United States|


**Analysis of Liquid Clustering:**

1.  Run the query on the liquid clustered table.
2.  Examine the Spark UI. Liquid Clustering provides similar data skipping benefits as Z-Ordering but is more adaptive to changes in your data and queries over time.

## Note the results for comparison
| Metric | Raw | Partitioned | Z-order | Liquid |
|--------|-------|--------|--------|-------|
| Files scanned | ? | ? | ? | ? |
| Bytes read | ? | ? | ? | ? |
| Duration (s) | ? | ? | ? | ? |

# Part 2: Handling small and orphaned files 

## Technique 1: Manual Compaction (OPTIMIZE)

Small files inflate job overhead (task scheduling, metadata reads, open/close costs) and can limit effective predicate pushdown (more file headers to scan). `OPTIMIZE` rewrites many small files into fewer larger ones (Databricks default target ~256MB unless overridden) without changing logical content.

### What It Solves
- Reduces per-task scheduling overhead
- Improves scan & filter efficiency by reducing file count
- Prepares table for better Z-Ordering / clustering effectiveness

### When to Use
- Many small files (rule of thumb: lots of files < 100MB)
- Bursty or streaming ingestion producing tiny batch outputs
- Before heavy analytical / BI workloads to lower latency

### When NOT to Use
- Files already in healthy size band (~100MB–1GB depending on workload)
- Ultra-low latency streaming where rewrite cost is disruptive
- Limited compute window / cost constraints (schedule off-peak instead)

> Rule of thumb: Target 100–500MB average file size for large analytical tables. Monitor via `DESCRIBE DETAIL` (numFiles & sizeInBytes / numFiles).

### Simulation Approach
We append tiny random samples repeatedly to manufacture fragmentation, then run `OPTIMIZE` to compact.


Run loop to append small data.

In [0]:
# Simulate small file creation by writing in a loop
for i in range(10):
    (df_raw.sample(fraction=0.001)
     .write.format("delta")
     .mode("append")
     .saveAsTable(tbl('to_compact')))

List number of files

In [0]:
# Check the number of files
spark.sql(f"DESCRIBE DETAIL {tbl('to_compact')}").select("numFiles").show()

+--------+
|numFiles|
+--------+
|      17|
+--------+



Execute `OPTIMIZE` (compaction only — no Z-order here).

In [0]:
# Now, perform manual compaction
delta_table_to_compact = DeltaTable.forName(spark, tbl('to_compact'))
delta_table_to_compact.optimize().executeCompaction()

DataFrame[path: string, metrics: struct<autoCompactParallelismStats:void,clusteringMetrics:void,clusteringStats:void,deletionVectorStats:struct<numDeletionVectorRowsRemoved:bigint,numDeletionVectorsRemoved:bigint>,endTimeMs:bigint,fileAgeHistogram:void,filesAdded:struct<avg:double,max:bigint,min:bigint,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<avg:double,max:bigint,min:bigint,totalFiles:bigint,totalSize:bigint>,numBatches:bigint,numBins:bigint,numBytesSkippedToReduceWriteAmplification:bigint,numFilesAdded:bigint,numFilesRemoved:bigint,numFilesSkippedToReduceWriteAmplification:bigint,numTableColumns:bigint,numTableColumnsWithStats:bigint,partitionsOptimized:bigint,preserveInsertionOrder:boolean,recompressionCodec:void,skippedArchivedFiles:bigint,startTimeMs:bigint,totalClusterParallelism:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,totalScheduledTasks:bigint,totalTaskExecutionTimeMs:bigint,zOrderStats:void>]

Re-list object count

In [0]:
# Check the number of files
spark.sql(f"DESCRIBE DETAIL {tbl('to_compact')}").select("numFiles").show()

+--------+
|numFiles|
+--------+
|       1|
+--------+



**Analysis of Compaction:**

- Observe the reduction in the number of files after running `OPTIMIZE`. This leads to more efficient reads as Spark needs to open and process fewer files.

## Technique 2: Auto Compaction

Auto Optimize (optimizeWrite + autoCompact) reduces operational toil: it sizes files during write and schedules asynchronous compaction of lingering small files.

### Components
- `delta.autoOptimize.optimizeWrite`: coalesces output partitions at commit time
- `delta.autoOptimize.autoCompact`: background merge of residual small files after commit

### When to Use
- Streaming pipelines or frequent micro-batch ingestion
- Continuous incremental loads where manual scheduling adds complexity
- Teams lacking bandwidth for manual compaction orchestration

### When NOT to Use
- Large, infrequent batch jobs already producing good-sized files
- Cost-sensitive environments (background merges add compute)
- Need for bespoke compaction policies / ordering (custom logic required)

### Caveats
- Background lag: immediate file count may remain high briefly
- Not a substitute for Z-Ordering / Liquid Clustering (does not cluster for skipping)
- Does not fix skew or suboptimal partition strategy

Auto Compaction automatically triggers compaction during writes without manual intervention; you primarily verify its effect via file counts and table history (each micro-batch ideally yields a handful of well-sized files instead of many tiny ones).


Create table with properties enabled.


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {tbl('auto_compact')}")

spark.sql(f"""
CREATE TABLE {tbl('auto_compact')}
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)
AS SELECT * FROM {tbl('raw')}
""")


DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

Check the number of files.

You can see that the original table contained hundreds/thousands of small files, but the auto_compact table contains now just 1 (if you run this notebook for the first time)

In [0]:
spark.sql(f"DESCRIBE DETAIL {tbl('raw')}").select("numFiles").show()
spark.sql(f"DESCRIBE DETAIL {tbl('auto_compact')}").select("numFiles").show()

+--------+
|numFiles|
+--------+
|   10000|
+--------+

+--------+
|numFiles|
+--------+
|       4|
+--------+



See the number of rows before performing many small appends

In [0]:
spark.sql(f"SELECT count(*) FROM {tbl('auto_compact')}").show()

+--------+
|count(*)|
+--------+
|50000000|
+--------+



Perform 55 small appends (approx 250-500 rows per batch)

In [0]:
for i in range(55):
    (df_raw.sample(fraction=0.00001)
     .write.format("delta")
     .mode("append")
     .saveAsTable(tbl('auto_compact')))

Count the number of rows to confirm that new data was appended

In [0]:
spark.sql(f"SELECT count(*) FROM {tbl('auto_compact')}").show()

+--------+
|count(*)|
+--------+
|50027661|
+--------+



**Analysis of Auto Compaction:**

Delta Lake's auto-optimization features manage file sizes during the write phase, preventing the creation of many small files. To check how many files were written during each write, inspect the table's history.

1. Query the table's history using `DESCRIBE HISTORY`.
2. Look at the `operation` and `operationMetrics` columns.
3. Notice that each WRITE operation results in just one new file (`"numFiles":"1"`).

The `delta.autoOptimize.optimizeWrite` property efficiently handles file sizes as data is written. The `autoCompact` feature triggers a separate OPTIMIZE operation only if many small files accumulate. Since `optimizeWrite` is managing this, you might not see separate compaction steps in the history.

To verify:
- Browse the history and check the number of files written during each write.
- You should find the OPTIMIZE command if autoCompact was triggered.

Note: Immediately after a write, the file count may still be high briefly. Re-run the query after some time to see the difference.

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {tbl('auto_compact')}"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
56,2026-06-07T05:57:17.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),a6db944e-2cfb-4ac3-a3d1-58084b4db4f4,0607-051857-9z86skth-v2n,55,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 535, numOutputBytes -> 16373)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
55,2026-06-07T05:57:11.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),e2fbe0bc-0b7c-4227-99e9-c2e9fd571fcc,0607-051857-9z86skth-v2n,54,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 493, numOutputBytes -> 15001)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
54,2026-06-07T05:57:06.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),6de98f40-1407-4bf0-993b-2106c9400157,0607-051857-9z86skth-v2n,53,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 499, numOutputBytes -> 15479)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
53,2026-06-07T05:57:00.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),98debabb-6ea7-4aa5-a70d-97b4ffe3aa1e,0607-051857-9z86skth-v2n,52,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 493, numOutputBytes -> 15285)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
52,2026-06-07T05:56:55.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),4b30bc3c-3062-4c66-8f90-c1ac3f3f46af,0607-051857-9z86skth-v2n,51,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 500, numOutputBytes -> 15151)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
51,2026-06-07T05:56:48.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1711045808821474),f3082354-da5d-4905-b5ad-39765340169d,0607-051857-9z86skth-v2n,50,SnapshotIsolation,false,"Map(numRemovedFiles -> 50, numRemovedBytes -> 765007, p25FileSize -> 555852, numDeletionVectorsRemoved -> 0, minFileSize -> 555852, numAddedFiles -> 1, maxFileSize -> 555852, p75FileSize -> 555852, p50FileSize -> 555852, numAddedBytes -> 555852)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
50,2026-06-07T05:56:43.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),f3082354-da5d-4905-b5ad-39765340169d,0607-051857-9z86skth-v2n,49,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 491, numOutputBytes -> 14942)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
49,2026-06-07T05:56:36.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),4a6e0085-2370-40ba-93f9-b904b1fe807c,0607-051857-9z86skth-v2n,48,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 481, numOutputBytes -> 15060)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
48,2026-06-07T05:56:30.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(1711045808821474),3d384f4c-7c5a-40fb-ab38-a6509f1c3cff,0607-051857-9z86skth-v2n,47,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 496, numOutputBytes -> 15061)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
47,2026-06-07T05:56:23.000Z,1664888573231191,shrikantpandeymnnit2015@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad 

Check number of files

In [0]:
spark.sql(f"DESCRIBE DETAIL {tbl('auto_compact')}").select("numFiles").show()

+--------+
|numFiles|
+--------+
|      10|
+--------+



## Technique 3: VACUUM

`VACUUM` removes obsolete files (not referenced by the current Delta log) to reclaim storage & trim metadata. It does NOT rewrite active data; it purges old snapshots beyond retention.

### Safety & Retention
- Default retention: 7 days (168h) to protect time travel & long-running readers
- Never shorten in production unless you fully understand recovery/RPO requirements

### When to Use
- After major rewrite / delete / update operations producing many stale files
- Storage costs are material and retention window already satisfied
- Old versions no longer needed for compliance or time travel queries

### When NOT to Use
- Need historical versions for audits / debugging within retention horizon
- Active streaming or batch readers may still reference older snapshots
- Immediately after heavy DML when retention window not met

> Production Practice: Retain at least 7 days (often 14–30+ for regulated domains). Use table history & storage metrics to justify more frequent vacuuming.


Perform updates/deletes before VACUUM to create orphaned files.

In [0]:
# Example update: Increase quantity by 10 for a specific customer
spark.sql(f"""
UPDATE {tbl('partitioned')}
SET quantity = quantity + 10
WHERE customer_id >= 1500
""")

# Example delete: Remove records for a specific product
spark.sql(f"""
DELETE FROM {tbl('partitioned')}
WHERE product_id <= 250
""")

DataFrame[num_affected_rows: bigint]

Check number of files before vacuum

In [0]:
spark.sql(f"DESCRIBE DETAIL {tbl('partitioned')}").select("numFiles").show()

+--------+
|numFiles|
+--------+
|      20|
+--------+



Run VACUUM

> **Note:**  
> In Databricks Free Edition, you cannot disable the `retentionDurationCheck` safety feature.  
> This means you may need to wait up to **7 days** after performing update or delete operations before orphaned files can be removed by VACUUM.

In [0]:
# Disable the retention check
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# VACUUM the partitioned table
delta_table_to_vacuum = DeltaTable.forName(spark, tbl('partitioned'))
delta_table_to_vacuum.vacuum()

# Re-enable the retention check
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

Check number of files after vacuum

In [0]:
spark.sql(f"DESCRIBE DETAIL {tbl('partitioned')}").select("numFiles").show()

+--------+
|numFiles|
+--------+
|      20|
+--------+



**Analysis of VACUUM:**

- The `VACUUM` command cleans up files that are no longer part of the current version of the table. You can check the file system before and after running `VACUUM` on a table that has undergone several modifications to see the effect.

# Project Conclusion

You applied multiple, complementary Delta optimization techniques and observed their impact empirically.

(See preceding section: Common Mistakes & Anti-Patterns for operational guardrails.)

### Comparative Summary (Qualitative)
| Technique | Primary Benefit | Ideal Use Case | Ongoing Maintenance |
|-----------|-----------------|----------------|---------------------|
| Partitioning | Directory pruning | Low-cardinality, common filters | Low once chosen |
| Z-Ordering | Multi-column data skipping | Repeated predicates across moderate/high-card columns | Periodic re-run as data grows |
| Manual Compaction | Fewer larger files | Backfill / bursty small writes | Run as needed (monitor file size dist) |
| Auto Optimize | Reduce small files proactively | Continuous ingestion workloads | Minimal |
| Liquid Clustering | Adaptive layout | Dynamic predicates & evolving schemas | Managed (monitor clustering metrics) |
| VACUUM | Storage reclamation | Any table with churn | Scheduled (respect retention) |

### Key Heuristics
- Start simple: partition only when clear benefit
- Use Z-Order or Liquid Clustering to refine skipping without over-partitioning
- Monitor file counts & avg file size (DESCRIBE DETAIL) for compaction signals
- Track clustering freshness (last Z-Order or clustering maintenance)
- Align time travel retention with recovery & audit requirements

### Common Mistakes & Anti-Patterns

1. Over-partitioning modest datasets (creates thousands of tiny directories & files)
2. Z-Ordering on too many columns (shuffle blow-up; diminishing returns >4)
3. Skipping VACUUM in cost-sensitive environments (latent storage bloat)
4. Using Liquid Clustering on tiny tables (<1GB) where overhead > benefit
5. Running manual OPTIMIZE on tables already governed by auto compaction (duplicate cost)
6. Z-Ordering on extremely high-cardinality random IDs / high-precision timestamps (low locality gain)
7. Not monitoring average file size drift after optimizations (missing 100MB–1GB target band)
8. Blanket applying the same strategy to every table without considering access patterns
9. Forgetting to re-run Z-Order after large backfills (stale clustering)
10. Lowering VACUUM retention below recovery requirements (operational risk)

> Recommendation: Maintain a lightweight observability sheet (table, avg file size, numFiles, last OPTIMIZE/Z-ORDER run, retention) to proactively detect drift.

### Suggested Next Experiments
1. Add a date-based partition layer and compare with country partitioning
2. Introduce updates/deletes & measure impact on small file generation
3. Benchmark query patterns with and without Z-Order refresh after significant data growth
4. Use Photon runtime and compare scan metrics
5. Track metrics programmatically and plot improvements over time

## Cleanup
Run the following cell to drop the catalog and all associated tables created during this project.

In [0]:
# Drop the catalog and all its contents
#spark.sql(f"DROP CATALOG IF EXISTS {CATALOG_NAME} CASCADE")